# Titanic - Feature Engineering

Neste notebook vamos preparar os dados para modelagem:
- Tratar valores faltantes
- Criar novas features
- Fazer encoding de variáveis categóricas
- Normalizar/padronizar quando necessário
- Salvar datasets processados

## 1. Setup e Carregamento dos Dados

In [ ]:
# Importar bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Bibliotecas importadas com sucesso!")

In [ ]:
# Carregar datasets
train = pd.read_csv('dados/train.csv')
test = pd.read_csv('dados/test.csv')

# Guardar PassengerId para submissão
test_ids = test['PassengerId'].copy()

print(f"📊 Train: {train.shape}")
print(f"📊 Test: {test.shape}")
print(f"\n✅ Dados carregados!")

## 2. Análise de Valores Faltantes

In [ ]:
# Função para analisar valores faltantes
def check_missing(df, name):
    print(f"\n🔍 Valores Faltantes - {name}")
    print("="*60)
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    
    if len(missing) == 0:
        print("✅ Nenhum valor faltante!")
    else:
        missing_df = pd.DataFrame({
            'Coluna': missing.index,
            'Faltantes': missing.values,
            'Percentual (%)': (missing.values / len(df) * 100).round(2)
        })
        print(missing_df.to_string(index=False))

check_missing(train, 'TREINO')
check_missing(test, 'TESTE')

## 3. Feature Engineering

### 3.1. Extrair Título do Nome

In [ ]:
# Extrair título do nome
def extract_title(df):
    df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
    return df

train = extract_title(train)
test = extract_title(test)

print("👑 Títulos encontrados no TREINO:")
print(train['Title'].value_counts())
print("\n👑 Títulos encontrados no TESTE:")
print(test['Title'].value_counts())

In [ ]:
# Simplificar títulos raros
def simplify_titles(df):
    # Mapear títulos raros para categorias mais comuns
    title_mapping = {
        'Mr': 'Mr',
        'Miss': 'Miss',
        'Mrs': 'Mrs',
        'Master': 'Master',
        'Dr': 'Rare',
        'Rev': 'Rare',
        'Col': 'Rare',
        'Major': 'Rare',
        'Mlle': 'Miss',  # Mademoiselle = Miss em francês
        'Mme': 'Mrs',    # Madame = Mrs em francês
        'Don': 'Rare',
        'Dona': 'Rare',
        'Lady': 'Rare',
        'Countess': 'Rare',
        'Jonkheer': 'Rare',
        'Sir': 'Rare',
        'Capt': 'Rare',
        'Ms': 'Miss'
    }
    
    df['Title'] = df['Title'].map(title_mapping)
    # Se algum título não foi mapeado, colocar como 'Rare'
    df['Title'].fillna('Rare', inplace=True)
    return df

train = simplify_titles(train)
test = simplify_titles(test)

print("✅ Títulos simplificados!")
print("\nTREINO:")
print(train['Title'].value_counts())
print("\nTESTE:")
print(test['Title'].value_counts())

### 3.2. Criar Features de Família

In [ ]:
# Criar Family_Size e IsAlone
def create_family_features(df):
    # Tamanho da família (incluindo o passageiro)
    df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
    
    # Está sozinho?
    df['IsAlone'] = (df['Family_Size'] == 1).astype(int)
    
    return df

train = create_family_features(train)
test = create_family_features(test)

print("✅ Features de família criadas!")
print(f"\nFamily_Size - TREINO:")
print(train['Family_Size'].value_counts().sort_index())
print(f"\nIsAlone - TREINO:")
print(train['IsAlone'].value_counts())

### 3.3. Tratar Idade (Age)

In [ ]:
# Preencher Age com a mediana por Pclass e Title
def fill_age(df):
    # Calcular mediana de idade por Pclass e Title
    age_by_group = df.groupby(['Pclass', 'Title'])['Age'].transform('median')
    
    # Preencher valores faltantes
    df['Age'] = df['Age'].fillna(age_by_group)
    
    # Se ainda houver NaN (caso raro), preencher com mediana geral
    df['Age'].fillna(df['Age'].median(), inplace=True)
    
    return df

train = fill_age(train)
test = fill_age(test)

print("✅ Idade preenchida!")
print(f"Valores faltantes em Age (TREINO): {train['Age'].isnull().sum()}")
print(f"Valores faltantes em Age (TESTE): {test['Age'].isnull().sum()}")

### 3.4. Criar Grupos de Idade (Age_Group)

In [ ]:
# Criar grupos de idade
def create_age_groups(df):
    df['Age_Group'] = pd.cut(df['Age'], 
                             bins=[0, 12, 18, 60, 100], 
                             labels=['Child', 'Teen', 'Adult', 'Senior'])
    return df

train = create_age_groups(train)
test = create_age_groups(test)

print("✅ Grupos de idade criados!")
print("\nTREINO:")
print(train['Age_Group'].value_counts().sort_index())

### 3.5. Tratar Embarked

In [ ]:
# Preencher Embarked com a moda (valor mais frequente)
def fill_embarked(df):
    df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
    return df

train = fill_embarked(train)
test = fill_embarked(test)

print("✅ Embarked preenchido!")
print(f"Valores faltantes em Embarked (TREINO): {train['Embarked'].isnull().sum()}")
print(f"Valores faltantes em Embarked (TESTE): {test['Embarked'].isnull().sum()}")